In [93]:
import datetime
import ta
import numpy as np
import yfinance as yf
from datetime import datetime as dt
import datetime as date
import seaborn as sn
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (20,10)
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [94]:
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier

In [95]:
# download csv from url
url = 'https://archives.nseindia.com/content/indices/ind_nifty100list.csv'
nifty100 = pd.read_csv(url)
nifty100.to_csv('ind_nifty100list.csv', index=False)

In [96]:
symbol= pd.read_csv("ind_nifty100list.csv")
symbol["ticker"] = symbol["Symbol"] + ".NS"
ticker= symbol['ticker'].values.tolist()
pricechg = symbol["Symbol"] + " pricechg"
percentchg = symbol["Symbol"] #+ " percentchg"
pricechg= pricechg.values.tolist()
percentchg= percentchg.values.tolist()
sd = yf.download(ticker, interval='1d')
data= sd['Adj Close']

[*********************100%***********************]  100 of 100 completed


In [97]:
# # This coding block requires inputs for each of the variables as described below-
# # returnperiod- No. of months over which the returns of stocks is measured to create a portfolio
# # holdingperiod- Following no. of months over which the stocks in the portfolio will be held for
# # sdate_input- Starting date from where the data for each stock should start (Starting date must fall on a Monday)
# # fdate_input- Ending date for where the data for each stock should end (Ending date must fall on a Friday)
# # input for dates should be in the following format -> "YYYY-MM-DD"
# returnperiod= 3
# holdingperiod= 6
# sdate_input= "2021-1-4"
# fdate_input= "2023-1-3"

In [122]:
# This coding block requires inputs for each of the variables as described below-
# returnperiod- No. of months over which the returns of stocks is measured to create a portfolio
# holdingperiod- Following no. of months over which the stocks in the portfolio will be held for
# sdate_input- Starting date from where the data for each stock should start (Starting date must fall on a Monday)
# fdate_input- Ending date for where the data for each stock should end (Ending date must fall on a Friday)
# input for dates should be in the following format -> "YYYY-MM-DD"
returnperiod= 3
holdingperiod= 6
# totalmonth = returnperiod +holdingperiod
# total_days = (totalmonth+1) *30
# get last 3 months and year
last3rd_month = datetime.date.today().replace(day=1) - datetime.timedelta(days=180)
sdate_input= last3rd_month.replace(day=1)
sdate_input = sdate_input + datetime.timedelta(days=-sdate_input.weekday(), weeks=1)

fdate_input= datetime.date.today().replace(day=1) - datetime.timedelta(days=33)
fdate_input = fdate_input + datetime.timedelta(days=4-fdate_input.weekday(), weeks=0)

sdate_input = sdate_input.strftime("%Y-%m-%d")
fdate_input = fdate_input.strftime("%Y-%m-%d")
print(sdate_input, fdate_input)

2022-11-07 2023-03-31


In [121]:
dt = datetime.date.today().replace(day=1)
dt

dt2= datetime.timedelta(days=180)
dt2

datetime.timedelta(days=180)

In [99]:
# start= dt.strptime(sdate_input, '%Y-%m-%d')
# finish= dt.strptime(fdate_input,'%Y-%m-%d')
#
# forw = data.loc[start:finish]
# dailydf = data.loc[start:finish]
# forw.reset_index(inplace=True)
# forw['date_num'] = forw['Date'].dt.weekday
# weeklydf = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]
# weeklydfcopy = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]
# returndf = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]
# print(start,finish)

In [123]:
import pytz

start = pytz.utc.localize(dt.strptime(sdate_input, '%Y-%m-%d'))
finish = pytz.utc.localize(dt.strptime(fdate_input,'%Y-%m-%d'))

forw = data.loc[start:finish]
dailydf = data.loc[start:finish]
forw.reset_index(inplace=True)
forw['date_num'] = forw['Date'].dt.weekday
weeklydf = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]
weeklydfcopy = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]
returndf = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]


AttributeError: 'datetime.date' object has no attribute 'strptime'

In [101]:
weeklydfcopy.set_index(np.arange(len(weeklydfcopy.index)), inplace=True, drop=True)

In [102]:
#Calculating return of stocks depending on the number of months input in "returnperiod" variable
#for creating porfolios 
for i,j in zip(returndf.index,weeklydfcopy.index):
    if j == weeklydfcopy.index[-1]:
        break
    if weeklydfcopy.loc[j,"date_num"]==0:
        if weeklydfcopy.loc[j+1,"date_num"]==0:
            #print(i)
            d = returndf.loc[i,"Date"] + datetime.timedelta(days=3)
            returndf.loc[i+1,"Date"]= d
            for k in forw.index:
                if forw.loc[k,"Date"]==d:
                    for tick in ticker:
                        returndf.loc[i+1,tick]= forw.loc[k,tick]
                        returndf.loc[i+1,"date_num"]=4
    
returndf.sort_index(inplace=True)
returndf.set_index(np.arange(1,len(returndf.index)+1,1),inplace=True, drop=True)

for i in returndf.index:
    if i==1:
        continue
    if i%8==0:
        continue
    if i%8==1:
        continue
    else:
        returndf.drop(i,inplace=True)

for i,j,k in zip(ticker, pricechg, percentchg):
    
   returndf[j]=returndf[i]-returndf[i].shift(returnperiod*2-1)
   returndf[k]=returndf[j]/returndf[i].shift(returnperiod*2-1)*100
   
returndf = returndf[returndf["date_num"]==4]
returndf.set_index(np.arange(1,len(returndf.index)+1,1), inplace=True, drop=True)

In [109]:
returndf
returndf.to_csv("x.csv")

In [104]:
#Portfolio formation
winner_portfolio= pd.DataFrame()
loser_portfolio= pd.DataFrame() 

# Portfolio of top 10 stocks made
for i in returndf.index:

    winner_portfolio.loc[i,'Date'] = returndf.loc[i,"Date"]

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W1']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W2']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W3']= j
            returndf.loc[i,j]= None
    
    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W4']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W5']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W6']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W7']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W8']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W9']= j
            returndf.loc[i,j]= None

    max= returndf.loc[i,percentchg].max()
    for j in percentchg:
        if returndf.loc[i,j]== max:
            winner_portfolio.loc[i,'W10']= j
            returndf.loc[i,j]= None

# Portfolio of bottom 10 stocks made
for i in returndf.index:

    loser_portfolio.loc[i,'Date'] = returndf.loc[i,"Date"] 

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L1']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L2']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L3']= j
            returndf.loc[i,j]= None
    
    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L4']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L5']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L6']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L7']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L8']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L9']= j
            returndf.loc[i,j]= None

    min= returndf.loc[i,percentchg].min()
    for j in percentchg:
        if returndf.loc[i,j]== min:
            loser_portfolio.loc[i,'L10']= j
            returndf.loc[i,j]= None


winner_portfolio.dropna(inplace=True)
winner_portfolio.reset_index(inplace=True)
del winner_portfolio["index"]

winner_weightage = pd.DataFrame()

winner_column_list=winner_portfolio.columns.values.tolist()
del winner_column_list[0]

loser_portfolio.dropna(inplace=True)
loser_portfolio.reset_index(inplace=True)
del loser_portfolio["index"]

loser_weightage = pd.DataFrame()

loser_column_list=loser_portfolio.columns.values.tolist()
del loser_column_list[0]

In [105]:
#Calculating weightage for winner portfolio using mean variance optimization
#To check the weightage dataframe print "winner_weightage" variable

for num in winner_portfolio.index:
    for d in dailydf.index:
        df = pd.DataFrame()
        if winner_portfolio.loc[num,"Date"] == d:
            list=winner_portfolio.loc[num,:].values.tolist()
            del list[0]
            for i in np.arange(len(list)):
                list[i]= list[i]+".NS"
            df = dailydf.loc[d- date.timedelta(days=70):d, list]
            mu = mean_historical_return(df)
            S = CovarianceShrinkage(df).ledoit_wolf()
            ef = EfficientFrontier(mu, S)#,weight_bounds=(-1,1))
            winner_weights = ef.max_sharpe()
            winner_cleaned_weights = ef.clean_weights()
            w=pd.Series(winner_cleaned_weights)
            for lm,ln in zip(list,winner_column_list):
                winner_weightage.loc[num,ln]= w[lm]

In [106]:
#Calculating return of stocks depending on the number of months input in "holdingperiod" variable
#for backtesting poertfolio returns

holdingdf = forw[(forw['date_num']== 0) | (forw['date_num']== 4)]

for i,j in zip(holdingdf.index,weeklydfcopy.index):
    if j == weeklydfcopy.index[-1]:
        break
    if weeklydfcopy.loc[j,"date_num"]==0:
        if weeklydfcopy.loc[j+1,"date_num"]==0:
            #print(i)
            d = holdingdf.loc[i,"Date"] + date.timedelta(days=3)
            holdingdf.loc[i+1,"Date"]= d
            for k in forw.index:
                if forw.loc[k,"Date"]==d:
                    for tick in ticker:
                        holdingdf.loc[i+1,tick]= forw.loc[k,tick]
                        holdingdf.loc[i+1,"date_num"]=4    

holdingdf.sort_index(inplace=True)
holdingdf.set_index(np.arange(1,len(holdingdf.index)+1,1),inplace=True, drop=True)

for i in holdingdf.index:
    if i==1:
        continue
    if i%8==0:
        continue
    if i%8==1:
        continue
    else:
        holdingdf.drop(i,inplace=True)

for i,j,k in zip(ticker, pricechg, percentchg):
    
   holdingdf[j]=holdingdf[i]-holdingdf[i].shift(holdingperiod*2-1)
   holdingdf[k]=holdingdf[j]/holdingdf[i].shift(holdingperiod*2-1)*100
   #monthly3df[k+"rsquared"] =monthly3df[k]*monthly3df[k]
   
holdingdf = holdingdf[holdingdf['date_num']== 4]
holdingdf.set_index(np.arange(1,len(holdingdf.index)+1,1), inplace=True, drop=True)
monthly=pd.DataFrame()
holdingdf.drop(np.arange(1,returnperiod+holdingperiod,1),inplace=True)
holdingdf.reset_index(inplace=True)
del holdingdf["index"]

In [107]:
#Calculating Backtest returns of winner portfolio
#"winner_test_return" dataframe provides backtested returns 
# for each stock in each of the winner portfolios, for the period of time input in "holdingperiod" return variable
winner_test_return= pd.DataFrame()

for ln in winner_column_list:
    for w,j in zip(winner_portfolio[ln],holdingdf.index):
        k= holdingdf.columns.get_loc(w)
        winner_test_return.loc[j,ln]= holdingdf.iloc[j,k] 
        if j == holdingdf.index[-1]:
            break

winner_test_return.reset_index(inplace=True)
del winner_test_return["index"]
   
winner_weighted_return= pd.DataFrame()

for ln in winner_column_list:
    winner_weighted_return[ln]= winner_test_return[ln]*winner_weightage[ln]

winner_weighted_return.dropna(inplace=True)

#"winner_weighted_return["Sum"]" dataframe provides total weighted average return for each winner portfolio
winner_weighted_return["Sum"]= winner_weighted_return.loc[:,winner_column_list].sum(axis=1)

KeyError: 'W1'

In [ ]:
#Calculating Backtest returns of loser portfolio
#"loser_test_return" dataframe provides backtested returns 
# for each stock in each of the loser portfolios, for the period of time input in "holdingperiod" return variable

loser_test_return= pd.DataFrame()

for ln in loser_column_list:
    for w,j in zip(loser_portfolio[ln],holdingdf.index):
        k= holdingdf.columns.get_loc(w)
        loser_test_return.loc[j,ln]= holdingdf.iloc[j,k] 
        if j == holdingdf.index[-1]:
            break

loser_test_return.reset_index(inplace=True)
del loser_test_return["index"]

loser_weighted_return= pd.DataFrame()

#"loser_weighted_return["Sum"]" dataframe provides total weighted average return for each loser portfolio
loser_weighted_return["Sum"]= loser_test_return.mean(axis=1)

In [ ]:
#Output gives the total return of winner portfolio for each churning period
winner_weighted_return["Sum"]

In [ ]:
#Output gives the total return of loser for each churning period
loser_weighted_return["Sum"]

In [ ]:
avg=winner_weighted_return["Sum"].mean()
std=winner_weighted_return["Sum"].std()
print("Winner Portfolio mean return:",avg)
print("Winner Portfolio Std. dev.:", std)

In [ ]:
avg=loser_weighted_return["Sum"].mean()
std=loser_weighted_return["Sum"].std()
print("Loser Portfolio mean return:",avg)
print("Loser Portfolio Std. dev.:", std)

In [ ]:
sn.set_theme()
plt.title('Winner Portfolio plot')
plt.xlabel('No. of months')
plt.ylabel('Return in each churn')
plt.yticks(np.arange(-50,100,step=5))
plt.xticks(np.arange(0,100,step=1))
plt.scatter(winner_weighted_return.index, winner_weighted_return["Sum"])
plt.show()

In [ ]:
sn.set_theme()
plt.title('Loser Portfolio plot')
plt.xlabel('No. of months')
plt.ylabel('Return in each churn')
plt.yticks(np.arange(-50,100,step=5))
plt.xticks(np.arange(0,100,step=1))
plt.scatter(loser_weighted_return.index, loser_weighted_return["Sum"])
plt.show()

In [ ]:
winner_portfolio.to_csv("01_winner_portfolio.csv", index=False)
loser_portfolio.to_csv("01_loser_portfolio.csv", index=False)

In [ ]:
winner_portfolio

In [ ]:
loser_portfolio

In [ ]:
holdingdf

## Parabolic SAR Strategy

In [ ]:
winner_portfolio = winner_portfolio.tail(1)
loser_portfolio = loser_portfolio.tail(1)
start_date = pd.to_datetime(winner_portfolio.Date.iloc[-1]).date()
before_start_date = start_date - date.timedelta(days=30)
# print(winner_portfolio,loser_portfolio,start_date,before_start_date)

In [ ]:
def get_trades_PSAR(portfolio,winner=True):
    Profit_df = pd.DataFrame()
    portfolio = portfolio.values.tolist()[0][1:]
    portfolio = [ i + '.NS' for i in portfolio]
    data = yf.download(portfolio,start= before_start_date)
    for stock in portfolio:
        data[f'PSAR {stock}'] = ta.trend.PSARIndicator(high=data['High'][stock],low=data['Low'][stock],close=data['Close'][stock]).psar()
        # drop first two rows
        data[f'Up/Down {stock}'] = data[f'PSAR {stock}'] - data['Close'][stock]
        data[f'Up/Down {stock}'] = data[f'Up/Down {stock}'].apply(lambda x: 1 if x>0 else -1)
    data.index = data.index.date
    data= data[data.index >= start_date]
    if winner:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == -1:
                try:
                    index = data[data[f'Up/Down {stock}'] == 1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = end_price - start_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'PSAR','Profit':profit,'Return':profit/start_price},index=[0])])
    else:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == 1:
                try:
                    index = data[data[f'Up/Down {stock}'] == -1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = start_price - end_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'PSAR','Profit':profit,'Return':profit/start_price},index=[0])])

    Profit_df.reset_index(inplace=True,drop=True)
    return Profit_df


In [ ]:
winners_profit_PSAR = get_trades_PSAR(winner_portfolio)
losers_profit_PSAR = get_trades_PSAR(loser_portfolio,winner=False)

In [ ]:
winners_profit_PSAR.to_csv('winners_profit_PSAR.csv',index=False)
losers_profit_PSAR.to_csv('losers_profit_PSAR.csv',index=False)

In [ ]:
winners_profit_PSAR

In [ ]:
losers_profit_PSAR

## SMA Strategy

In [ ]:
def get_trades_SMA(portfolio,winner=True):
    Profit_df = pd.DataFrame()
    portfolio = portfolio.values.tolist()[0][1:]
    portfolio = [ i + '.NS' for i in portfolio]
    data = yf.download(portfolio,start= before_start_date)
    for stock in portfolio:
        data[f'SMA {stock}'] = ta.trend.SMAIndicator(close=data['Close'][stock],window=10).sma_indicator()
        # drop first two rows
        data[f'Up/Down {stock}'] = data[f'SMA {stock}'] - data['Close'][stock]
        data[f'Up/Down {stock}'] = data[f'Up/Down {stock}'].apply(lambda x: 1 if x>0 else -1)
    data.index = data.index.date
    data= data[data.index >= start_date]
    if winner:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == -1:
                try:
                    index = data[data[f'Up/Down {stock}'] == 1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = end_price - start_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'SMA','Profit':profit,'Return':profit/start_price},index=[0])])

    else:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == 1:
                try:
                    index = data[data[f'Up/Down {stock}'] == -1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = start_price - end_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'SMA','Profit':profit,'Return':profit/start_price},index=[0])])
    Profit_df.reset_index(inplace=True,drop=True)
    return Profit_df

In [ ]:
winners_profit_SMA = get_trades_SMA(winner_portfolio)
loser_porfit_SMA = get_trades_SMA(loser_portfolio,winner=False)

In [ ]:
loser_porfit_SMA

In [ ]:
winners_profit_SMA

## EMA Strategy

In [ ]:
def get_trades_EMA(portfolio,winner=True):
    Profit_df = pd.DataFrame()
    portfolio = portfolio.values.tolist()[0][1:]
    portfolio = [ i + '.NS' for i in portfolio]
    data = yf.download(portfolio,start= before_start_date)
    for stock in portfolio:
        data[f'EMA {stock}'] = ta.trend.EMAIndicator(close=data['Close'][stock],window=10).ema_indicator()
        # drop first two rows
        data[f'Up/Down {stock}'] = data[f'EMA {stock}'] - data['Close'][stock]
        data[f'Up/Down {stock}'] = data[f'Up/Down {stock}'].apply(lambda x: 1 if x>0 else -1)
    data.index = data.index.date
    data= data[data.index >= start_date]
    if winner:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == -1:
                try:
                    index = data[data[f'Up/Down {stock}'] == 1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = end_price - start_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'EMA','Profit':profit,'Return':profit/start_price},index=[0])])

    else:
        for stock in portfolio:
            # if PSAR is -1
            if data[f'Up/Down {stock}'].values[0] == 1:
                try:
                    index = data[data[f'Up/Down {stock}'] == -1].index[0]
                except:
                    index = data.tail(1).index[0]
                start_price = float(data.iloc[0]['Close'][stock])
                end_price = float(data[data.index == index]['Close'][stock])
                profit = start_price - end_price
                Profit_df = pd.concat([Profit_df,pd.DataFrame({'Stock':stock,'Start_date':start_date,'End_date':index,'strategy':'EMA','Profit':profit,'Return':profit/start_price},index=[0])])
    Profit_df.reset_index(inplace=True,drop=True)
    return Profit_df

winners_profit_EMA = get_trades_EMA(winner_portfolio)
losers_profit_EMA = get_trades_EMA(loser_portfolio,winner=False)

In [ ]:
winners_profit_EMA


In [ ]:
losers_profit_EMA